# 02 · Sim-verified data generation (GPU runtime)

Each human demo is replayed in many **fresh** LIBERO scenes (never the 50 evaluation scenes), with the bowl/plate shifted by up to ±3 cm and the human's grasp and place points perturbed. LIBERO decides success.
- Successes become the imitation dataset.
- Failures are kept for the world model and for offline RL.

Then three LeRobot datasets are built through the same writer, so they differ only in their data:

| dataset | content |
|---|---|
| `ours_all` | all successful generated episodes |
| `ours_50` | 50 successes spread evenly over the human demos (matched count) |
| `official` | the official teleop demos of this task from `lerobot/libero`, resized to 128 px |

Runtime: ~1–1.5 h on a T4. Generation is resumable: if Colab disconnects, re-run and it continues.

In [ ]:
import os, sys, json, subprocess
os.environ["MUJOCO_GL"] = "egl"                      # GPU rendering, must be set before mujoco is imported
REPO_URL = "https://github.com/LukasWill/phone2panda.git"
CODE_DIR = "/content/phone2panda" if os.path.exists("/content") else "/kaggle/working/phone2panda"
if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lerobot[libero]==0.6.1", "imageio-ffmpeg"], check=True)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
# Hugging Face token (Colab: key icon -> Secrets -> HF_TOKEN; Kaggle: Add-ons -> Secrets -> HF_TOKEN)
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN found: uploads to the Hub will fail")
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
PER_DEMO = 40                  # rollouts per human demo (20 demos -> 800 episodes)
STRATEGY = "object_relative"   # best strategy from Table 1 (results/table1.csv)
WORKERS = 2
HUB_REPO = "LukasWill/phone2panda-data"   # private dataset repo on the Hugging Face Hub
GEN_DIR = "/content/generated" if os.path.exists("/content") else "/kaggle/working/generated"

In [ ]:
!python -m p2p.data.generate data/processed {GEN_DIR} --per-demo {PER_DEMO} --strategy {STRATEGY} --workers {WORKERS}

## Generation yield per human demo

In [ ]:
import glob, numpy as np, pandas as pd
rows = [json.loads(str(np.load(f)["meta"])) for f in sorted(glob.glob(f"{GEN_DIR}/ep_*.npz"))]
df = pd.DataFrame(rows)
print(f"{len(df)} episodes, {df.success.sum()} successes ({100 * df.success.mean():.0f}%)")
print(df[~df.success].last_waypoint.value_counts().to_string())
df.groupby("demo").success.agg(["mean", "count"]).round(2)

In [ ]:
!python -m p2p.data.lerobot_io ours     {GEN_DIR} /content/lerobot/ours_all
!python -m p2p.data.lerobot_io ours     {GEN_DIR} /content/lerobot/ours_50 --balanced 50
!python -m p2p.data.lerobot_io ours     {GEN_DIR} /content/lerobot/ours_mix --all
!python -m p2p.data.lerobot_io official -          /content/lerobot/official

## Upload everything to the Hub (training runs on Colab or Kaggle pull from there)

In [ ]:
from p2p.hub import push
for name in ("ours_all", "ours_50", "ours_mix", "official"):
    push(f"/content/lerobot/{name}", HUB_REPO, path_in_repo=f"lerobot/{name}")
!cd {GEN_DIR}/.. && tar -cf generated.tar generated
push(os.path.dirname(GEN_DIR), HUB_REPO, path_in_repo="raw")   # generated.tar: needed by the world model and IQL
df.to_csv("generation_summary.csv", index=False); print("done")